# ZCU216 loopback and multi-tile synchronization

Generate a tone, capture it through XM655 loopback, then repeat MTS and inspect
the relative phase of the connected ADC inputs. Complete the setup notebook first.

The converters run at **2 GS/s**. The RFDC uses **4× DAC interpolation** and
**4× ADC decimation**, so both complex sample buffers run at **500 MS/s**.
The NCO shifts the digital tone to RF and back.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from rfsoc_mts.overlay import Overlay

converter_rate_hz = 2e9
interpolation = decimation = 4
player_rate_hz = converter_rate_hz / interpolation
capture_rate_hz = converter_rate_hz / decimation
nco_mhz = 200
tone_bin = 64
amplitude_codes = 1024
capture_channels = ['adc_00']  # Add only channels with a physical loopback.
repeats = 5

## Load the overlay and configure the clocks

Both converters use tile 2 as the MTS reference. All four tiles are included.
The first capture checks the analogue path before the repeated MTS experiment.

In [ ]:
ol = Overlay(lmx_freq=500, lmk_freq=500, dac_mts_reftile=2, adc_mts_reftile=2,
             dac_mts_tiles=0b1111, adc_mts_tiles=0b1111, sync_on_init=False)
assert ol.sample_rate_mhz * 1e6 == converter_rate_hz
pll_locked = 2  # XRFdc PLL lock status code
assert ol.rfdc.adc_tiles[2].PLLLockStatus == pll_locked, 'ADC tile 2 PLL unlocked'
assert ol.rfdc.dac_tiles[2].PLLLockStatus == pll_locked, 'DAC tile 2 PLL unlocked'

channel_indices = [ol.signal_capture.bram_hierarchies.index(name)
                   for name in capture_channels]
sample_count = ol.signal_player.length // 2
tone_hz = tone_bin * player_rate_hz / sample_count
time_s = np.arange(sample_count) / player_rate_hz
signal = amplitude_codes * np.exp(2j * np.pi * tone_hz * time_s)
print(f'Baseband tone: {tone_hz / 1e6:.3f} MHz')
print(f'RF tone: {nco_mhz + tone_hz / 1e6:.3f} MHz')

In [ ]:
ol.configure_rfdcs(dac_freq=nco_mhz, adc_freq=nco_mhz)
ol.signal_player.disable()
ol.signal_player.load_signal(signal)
ol.signal_player.enable()
try:
    capture = ol.signal_capture.capture_adc()[channel_indices]
finally:
    ol.signal_player.disable()

window = slice(0, 512)
capture_time_us = np.arange(capture.shape[-1]) / capture_rate_hz * 1e6
fig, ax = plt.subplots(figsize=(9, 3))
for name, samples in zip(capture_channels, capture):
    ax.plot(capture_time_us[window], samples.real[window], label=name)
ax.set(xlabel='Capture time (µs)', ylabel='ADC code (I)', title='Loopback capture')
ax.legend()
plt.show()

## Repeat MTS and compare relative phase

Use two or more connected ADC inputs on different tiles for the relative-phase
plot. Each run measures phase relative to the first selected channel. Cable delays
can give a fixed offset. A repeatable result has little run-to-run phase variation.
Review signal amplitude before interpreting phase.

In [ ]:
tone_reference = np.exp(-2j * np.pi * tone_hz * capture_time_us * 1e-6)
tone_response = []
for _ in range(repeats):
    ol.mts.mts_sync(dac=nco_mhz, adc=nco_mhz)
    ol.signal_player.enable()
    try:
        samples = ol.signal_capture.capture_adc()[channel_indices]
    finally:
        ol.signal_player.disable()
    tone_response.append(np.mean(samples * tone_reference, axis=1))

tone_response = np.asarray(tone_response)
print('Mean tone magnitude (ADC codes):', np.mean(np.abs(tone_response), axis=0))
ol.mts.report_latency()

if len(capture_channels) > 1:
    relative_phase_deg = np.angle(tone_response[:, 1:] * tone_response[:, :1].conj(),
                                  deg=True)
    fig, ax = plt.subplots(figsize=(8, 3))
    for name, phases in zip(capture_channels[1:], relative_phase_deg.T):
        ax.plot(np.arange(1, repeats + 1), phases, marker='o', label=name)
    ax.set(xlabel='MTS run', ylabel='Relative phase (°)',
           title=f'Phase relative to {capture_channels[0]}')
    ax.legend()
    plt.show()
else:
    print('Add a loopback on another ADC tile to compare relative phase.')